In [1]:
import pymongo
import pandas as pd

# الاتصال بمونجو
client = pymongo.MongoClient("mongodb://localhost:27017/")
mydb = client['mydatabase']
coll=mydb["customers"]

# جلب البيانات
cursor = coll.find()
docs = list(cursor)

# تحويل لـ DataFrame
df = pd.DataFrame(docs)

# تحويل ObjectId لسترنج
df['_id'] = df['_id'].astype(str)

df


,_id,name,adress,address
0,68bcb09c1d378012d3c219e5,ahmed,baltim,NaN
1,68bcb0a71d378012d3c219e7,ahmed,baltim,NaN
2,68bcb1451d378012d3c219e9,ahmed,baltim,NaN
3,68bcb1481d378012d3c219eb,ahmed,baltim,NaN
4,68bcb1501d378012d3c219ee,ahmed,baltim,NaN
...,...,...,...,...
100,68bcbaa21d378012d3c21a5b,Vicky,NaN,Yellow Garden 2
101,68bcbaa21d378012d3c21a5c,Ben,NaN,Park Lane 38
102,68bcbaa21d378012d3c21a5d,William,NaN,Central st 954
103,68bcbaa21d378012d3c21a5e,Chuck,NaN,Main Road 989


هنكريت داتا جديده وفيها لست ونعرف هنحولها ازاي في الجداول 
-----------------------------------------------------------------

In [1]:
from pymongo import MongoClient
import pandas as pd

client = MongoClient("mongodb://localhost:27017/")
db = client['media_db']
coll = db['media']

docs = list(coll.find())
df_media = pd.DataFrame(docs)

df_media

# النتيجه 
# عمود thumbnails عبارة عن list of dicts (قائمة من قواميس).

# العمود ده مش مفهوم لـ Power BI أو SQL.

,_id,title,type,url,size_kb,uploaded_by,uploaded_at,tags,thumbnails
0,68bda985b0d4e7678644152e,sunrise.jpg,image,https://example.com/sunrise.jpg,230,user_1,2025-09-01 06:00:00,"[nature, sunrise]","[{'w': 200, 'h': 100, 'url': 'https://example...."
1,68bda99ab0d4e7678644152f,sunrise.jpg,image,https://example.com/sunrise.jpg,230,user_1,2025-09-01 06:00:00,"[nature, sunrise]","[{'w': 200, 'h': 100, 'url': 'https://example...."


In [3]:
# عشان نعمل جدول جديد json_normalizeعشان نحل المشكله دي هنستخدم 

from pandas import json_normalize

#_id  ونربطه ب thumbnails  هنعمل جدول جديد لل 
df_thumbs = json_normalize(
    docs,
    record_path='thumbnails',       # امسك القايمة thumbnails
    meta=['_id'],    # رجّع معاها الأعمدة دي كمرجع
    sep="_"
)


#_id  ونربطه ب tages  هنعمل جدول جديد لل 
df_tags=pd.json_normalize(
   docs,
   record_path=['tags'],
   meta=["_id"],
   
).rename(columns={"_id":"media_id",0:"tag"})

#بنمسح من الجدول الرئيسي  الاعمده الي احنا عملنها جداول منفصله 
#df_media=df.drop(columns=["tags","thumbnails"])


# نخلي media_id العمود الأول
cols = ["media_id"] + [c for c in df_tags.columns if c != "media_id"]
df_tags = df_tags[cols]
 
df_media


,_id,title,type,url,size_kb,uploaded_by,uploaded_at,tags,thumbnails
0,68bda985b0d4e7678644152e,sunrise.jpg,image,https://example.com/sunrise.jpg,230,user_1,2025-09-01 06:00:00,"[nature, sunrise]","[{'w': 200, 'h': 100, 'url': 'https://example...."
1,68bda99ab0d4e7678644152f,sunrise.jpg,image,https://example.com/sunrise.jpg,230,user_1,2025-09-01 06:00:00,"[nature, sunrise]","[{'w': 200, 'h': 100, 'url': 'https://example...."


 لداتا الي عملنها في الداتا بيز lood هنعمل  
 ----------------------------------------------

In [6]:
# الاول نبني فانكشن بحيث ننضف بيها الجداول عشان ينفع تتخزن في sql

import json
from bson import ObjectId

def clean_mongo_document(df):
    df = df.copy()

    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: str(x) if isinstance(x, ObjectId) else (
                json.dumps(x) if isinstance(x, (list, dict)) else x
            )
        )
    return df


# تنظيف كل جدول
df_media_clean = clean_mongo_document(df_media)
df_thumbnails_clean = clean_mongo_document(df_thumbs)
df_tags_clean = clean_mongo_document(df_tags)



In [7]:
import pyodbc
from sqlalchemy import create_engine



# نجهز connection string
connection_string = "mssql+pyodbc://@localhost\\SQLEXPRESS/MongoData?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"

engine = create_engine(connection_string)

# نخزن df_media
df_media_clean.to_sql("media", con=engine, if_exists="replace", index=False)

# نخزن df_thumbnails
df_thumbnails_clean .to_sql("thumbnails", con=engine, if_exists="replace", index=False)

# نخزن df_tags
df_tags_clean.to_sql("tags", con=engine, if_exists="replace", index=False)

4